# Pancreatic MatchMaker — Colab A100 Runner

1. **Clone** the repo (GitHub).
2. **Install** deps.
3. **Knobs** (seeds, `RAW_CSV`, hyperparameters).
4. **Copy `data/` from Google Drive** — My Drive root → `./data/` (chem + synergy CSVs).
5. **Preprocess** → processed TSVs.
6. **Splits** (LTO / LPO / …).
7. **Train** (`scripts/run_experiments.py`).
8. **Report**.

Optional: copy `results/` and `splits/` to Drive using the cell at the end (`SAVE_TO_DRIVE`).

Defaults: **min–max** norm, **uniform** weights.

**Runtime → Change runtime type → GPU** before training.


In [11]:
# Quick file check — run AFTER **Load codebase**
from pathlib import Path

for name in ('helper_funcs.py', 'MatchMaker.py'):
    p = Path(name)
    print(name, 'OK' if p.exists() else 'MISSING — run checkout cell first')

hf = Path('helper_funcs.py')
if hf.exists():
    t = hf.read_text(encoding='utf-8', errors='replace')
    print('helper_funcs.py: minmax scaling path:', "minmax" in t and "'minmax'" in t)
mm = Path('MatchMaker.py')
if mm.exists():
    t = mm.read_text(encoding='utf-8', errors='replace')
    print('MatchMaker.py: TerminateOnNaN in trainer:', 'TerminateOnNaN' in t)


helper_funcs.py OK
MatchMaker.py OK
helper_funcs.py: minmax scaling path: True
MatchMaker.py: TerminateOnNaN in trainer: True


In [2]:
import os
import shutil

# Cleanup before cloning (run once per fresh session).
# Must chdir out of the tree we delete — otherwise cwd becomes invalid and later git/subprocess calls fail.
os.chdir("/content")
shutil.rmtree("/content/matchmaker_test", ignore_errors=True)
print("Removed /content/matchmaker_test if it existed.")


Removed /content/matchmaker_test if it existed.


In [12]:
# --- Load codebase from GitHub ---
import os
import shutil
import subprocess
import time
from pathlib import Path

GIT_REPO_URL = "https://github.com/ENS491-Project-360-Team/ens_492.git"
GIT_BRANCH = "main"
CLONE_PARENT = Path("/content/matchmaker_test")

# If cwd is inside CLONE_PARENT, deleting it breaks git ("Unable to read current working directory").
os.chdir("/content")

shutil.rmtree(CLONE_PARENT, ignore_errors=True)

cmd = ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_REPO_URL, str(CLONE_PARENT)]
print("Running:", " ".join(cmd))
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
last = None
for attempt in range(1, 4):
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
    last = proc
    if proc.returncode == 0:
        break
    print(f"git clone failed (attempt {attempt}/3), stderr:\n{proc.stderr or proc.stdout or '(no output)'}")
    if attempt < 3:
        time.sleep(4)
if last.returncode != 0:
    raise RuntimeError(
        "git clone failed after 3 attempts. If stderr mentioned cwd, re-run this cell. "
        "Else: network hiccup / private repo (use https://TOKEN@github.com/...)."
    )

if (CLONE_PARENT / "ens_492" / "main.py").exists():
    WORKDIR = CLONE_PARENT / "ens_492"
elif (CLONE_PARENT / "main.py").exists():
    WORKDIR = CLONE_PARENT
else:
    raise FileNotFoundError("Clone OK but main.py not found under clone root or ens_492/.")

os.chdir(WORKDIR)
os.environ["MATCHMAKER_ROOT"] = str(WORKDIR.resolve())
print("Working directory:", Path.cwd())


Running: git clone --depth 1 --branch main https://github.com/ENS491-Project-360-Team/ens_492.git /content/matchmaker_test
Working directory: /content/matchmaker_test


In [5]:
# Optional: debug Colab paths (ignore if `import tensorflow` next cell works)
!pwd

/content/matchmaker_test


In [13]:
# 2) Install dependencies
!python -m pip install -q --upgrade pip
!python -m pip install -q numpy pandas scipy scikit-learn tensorflow

import tensorflow as tf
print('TF version:', tf.__version__)
!nvidia-smi

TF version: 2.20.0
Mon May  4 12:15:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

In [14]:
# 3) Configure experiment knobs
SEEDS = [42]
RUN_TRAINING = True   # set False to only test preprocessing/splits
GPU_DEVICES = '0'

# Classification threshold for converting regression score to class prediction
CLASSIFICATION_THRESHOLD = 0.0

# Raw data: path relative to repo root, or basename if the file lives in `data/`
# (Step A checks cwd first, then `data/<basename>` when `RAW_CSV` has no folder).
# Use the exact filename on disk (sheet export often has spaces).
RAW_CSV = 'synergy - comb - Combination data.csv'

# Training / preprocessing (passed to scripts/run_experiments.py -> main.py)
# Defaults: merged pipeline (Arda-style min-max + selectable weights).
NORM = 'minmax'   # alternatives: tanh_norm, norm, tanh
WEIGHT_MODE = 'uniform'   # uniform | q3_upweight | log
WEIGHT_ALPHA = 3.0   # q3_upweight only
LR = 1e-4
INPUT_DROPOUT = 0.2
DROPOUT = 0.5
BATCH_SIZE = 128
MAX_EPOCH = 1000
EARLYSTOP = 100

print('SEEDS:', SEEDS)
print('RUN_TRAINING:', RUN_TRAINING)
print('RAW_CSV:', RAW_CSV)
print('NORM:', NORM, '| WEIGHT_MODE:', WEIGHT_MODE, '| LR:', LR, '| BATCH_SIZE:', BATCH_SIZE)

# Backup to Drive (optional). Step 9 copies results/splits when SAVE_TO_DRIVE is True.
from pathlib import Path

SAVE_TO_DRIVE = False
DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/matchmaker_runs')



SEEDS: [42]
RUN_TRAINING: True
RAW_CSV: synergy - comb - Combination data.csv
NORM: minmax | WEIGHT_MODE: uniform | LR: 0.0001 | BATCH_SIZE: 128


In [15]:
# 4) Copy data from Google Drive (My Drive root → `./data/`)
#
# Upload those files to the top of "My Drive", then run this after **Load codebase**.

from pathlib import Path
import shutil

from google.colab import drive

drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
dest_dir = Path("data")
dest_dir.mkdir(parents=True, exist_ok=True)

required = [
    "drug1_chem.csv",
    "drug2_chem.csv",
    "cell_line_gex.csv",
    "synergy - comb - Combination data.csv",
]

for name in required:
    src = MYDRIVE / name
    if not src.is_file():
        raise FileNotFoundError(
            "Not on Drive at {!r}. Open Drive → confirm the name matches exactly, then re-run.\n"
            "List folder: !ls -la '/content/drive/MyDrive'".format(src)
        )
    shutil.copy2(src, dest_dir / name)
    print("OK:", dest_dir / name)

# Optional leftovers on your Drive (pipeline can ignore if you use generated splits)
for name in ("DrugCombinationData.tsv", "train_inds.txt", "val_inds.txt", "test_inds.txt"):
    src = MYDRIVE / name
    if src.is_file():
        shutil.copy2(src, dest_dir / name)
        print("OK (extra):", dest_dir / name)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK: data/drug1_chem.csv
OK: data/drug2_chem.csv
OK: data/cell_line_gex.csv
OK: data/synergy - comb - Combination data.csv
OK (extra): data/DrugCombinationData.tsv
OK (extra): data/train_inds.txt
OK (extra): data/val_inds.txt
OK (extra): data/test_inds.txt


In [16]:
# 5) Step A - preprocess pancreatic data (writes unfiltered + Bliss-variance–filtered)
# Variance filter: IQR rule on replicate Bliss variance per (pair, cell) — same spirit as meeting4_dataprocessing.ipynb (without quartile class labels).
# Pure Python (no shell heredoc) — avoids NameError on stray `PY` in Colab/Cursor.
import os
import subprocess
import sys
import pandas as pd
from pathlib import Path

_raw = Path(RAW_CSV)
_candidates = [_raw.resolve()]
if not _raw.is_absolute() and _raw.parent == Path("."):
    _candidates.append((Path("data") / _raw.name).resolve())
csv_path = next((p for p in _candidates if p.is_file()), None)
if csv_path is None:
    tried = ", ".join(str(p) for p in _candidates)
    raise FileNotFoundError(
        "Raw CSV not found. Tried: "
        + tried
        + ". Set `RAW_CSV` to the path under your working directory (e.g. `data/yourfile.csv`) "
        "or upload the file to `data/` and keep the basename in `RAW_CSV`."
    )
print("Using raw CSV:", csv_path)

r = subprocess.run(
    [sys.executable, "scripts/prepare_pancreatic_data.py", "--raw-csv", str(csv_path), "--out-dir", "data/processed"],
    capture_output=True,
    text=True,
)
print(r.stdout)
if r.stderr:
    print(r.stderr)
r.check_returncode()
print('data/processed:', os.listdir('data/processed'))
print('unfiltered shape:', pd.read_csv('data/processed/pancreatic_unfiltered.tsv', sep='\t').shape)
print('variance_filtered shape:', pd.read_csv('data/processed/pancreatic_variance_filtered.tsv', sep='\t').shape)


Using raw CSV: /content/matchmaker_test/data/synergy - comb - Combination data.csv
Wrote: data/processed/pancreatic_unfiltered.tsv
Wrote: data/processed/pancreatic_variance_filtered.tsv
Wrote: data/processed/data_card.md

data/processed: ['data_card.md', 'pancreatic_unfiltered.tsv', 'pancreatic_variance_filtered.tsv']
unfiltered shape: (9893, 11)
variance_filtered shape: (9164, 11)


In [10]:
# 6) Step B - generate split files (LTO/LPO/LCO/LODO/LDO)
import json
import os
import subprocess
import sys

seed_args = [str(s) for s in SEEDS]
subprocess.run(
    [sys.executable, 'scripts/generate_splits.py', '--input-tsv', 'data/processed/pancreatic_variance_filtered.tsv',
     '--outdir', 'splits', '--seeds'] + seed_args,
    check=True,
)
print('splits:', os.listdir('splits'))
with open('splits/leakage_report.json') as f:
    leak = json.load(f)
print('num leakage entries:', len(leak))
print('example:', leak[0])


splits: ['ldo', 'lodo', 'leakage_report.json', 'lto', 'lpo', 'lco']
num leakage entries: 5
example: {'pair_overlap': 321, 'drug_overlap': 57, 'cell_overlap': 30, 'n_train': 5866, 'n_test': 1832, 'split': 'lto', 'seed': 42, 'folder': 'splits/lto/seed_42'}


In [24]:
# 7) Step C - optional dry run to verify command matrix
import subprocess
import sys

seed_args = [str(s) for s in SEEDS]
cmd = (
    [sys.executable, 'scripts/run_experiments.py',
     '--project-root', '.', '--seeds'] + seed_args
    + ['--splits-root', 'splits',
       '--processed-root', 'data/processed',
       '--out-root', 'results/runs',
       '--gpu-devices', GPU_DEVICES,
       '--classification-threshold', str(CLASSIFICATION_THRESHOLD),
       '--norm', NORM,
       '--weight-mode', WEIGHT_MODE,
       '--weight-alpha', str(WEIGHT_ALPHA),
       '--lr', str(LR),
       '--input-dropout', str(INPUT_DROPOUT),
       '--dropout', str(DROPOUT),
       '--batch-size', str(BATCH_SIZE),
       '--max-epoch', str(MAX_EPOCH),
       '--earlystop', str(EARLYSTOP),
       '--dry-run']
)
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/run_experiments.py', '--project-root', '.', '--seeds', '42', '--splits-root', 'splits', '--processed-root', 'data/processed', '--out-root', 'results/runs', '--gpu-devices', '0', '--classification-threshold', '0.0', '--norm', 'minmax', '--weight-mode', 'uniform', '--weight-alpha', '3.0', '--lr', '0.0001', '--input-dropout', '0.2', '--dropout', '0.5', '--batch-size', '128', '--max-epoch', '1000', '--earlystop', '100', '--dry-run'], returncode=0)

In [ ]:
# 8) Step D - full training matrix
# This runs: {unfiltered, filtered} x {lto,lpo,lco,lodo,ldo} x seeds
# main.py also needs gitignored feature files under `data/` (drug descriptors + cell GEX) — same as local zip.
import subprocess
import sys
from pathlib import Path

if RUN_TRAINING:
    need = [
        Path("data/drug1_chem.csv"),
        Path("data/drug2_chem.csv"),
        Path("data/cell_line_gex.csv"),
    ]
    missing = [p for p in need if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            "Training requires descriptor/GEX CSVs in `data/` (not created by preprocessing). "
            "Copy from your full project `data/` (or unzip `data.zip`) into this repo on Colab. Missing:\n  "
            + "\n  ".join(str(p) for p in missing)
        )

    seed_args = [str(s) for s in SEEDS]
    cmd = (
        [sys.executable, "scripts/run_experiments.py",
         "--project-root", ".", "--seeds"] + seed_args
        + ["--splits-root", "splits",
           "--processed-root", "data/processed",
           "--out-root", "results/runs",
           "--gpu-devices", GPU_DEVICES,
           "--classification-threshold", str(CLASSIFICATION_THRESHOLD),
           "--norm", NORM,
           "--weight-mode", WEIGHT_MODE,
           "--weight-alpha", str(WEIGHT_ALPHA),
           "--lr", str(LR),
           "--input-dropout", str(INPUT_DROPOUT),
           "--dropout", str(DROPOUT),
           "--batch-size", str(BATCH_SIZE),
           "--max-epoch", str(MAX_EPOCH),
           "--earlystop", str(EARLYSTOP)]
    )
    print("Running:", " ".join(cmd))
    p = subprocess.run(cmd)
    if p.returncode != 0:
        raise RuntimeError(
            "run_experiments / main.py exited with {}. Scroll up in this cell for the Python/TF traceback.".format(
                p.returncode
            )
        )
else:
    print("RUN_TRAINING is False, skipping training.")


Running: /usr/bin/python3 scripts/run_experiments.py --project-root . --seeds 42 --splits-root splits --processed-root data/processed --out-root results/runs --gpu-devices 0 --classification-threshold 0.0 --norm minmax --weight-mode uniform --weight-alpha 3.0 --lr 0.0001 --input-dropout 0.2 --dropout 0.5 --batch-size 128 --max-epoch 1000 --earlystop 100


In [ ]:
# 9) Step E - build summary report
import subprocess
import sys
from pathlib import Path

if RUN_TRAINING:
    subprocess.run(
        [
            sys.executable,
            "scripts/build_report.py",
            "--summary-csv",
            "results/summary_metrics.csv",
            "--per-seed-csv",
            "results/per_split_seed_metrics.csv",
            "--out-md",
            "results/report.md",
        ],
        check=True,
    )
    res = Path("results")
    if res.is_dir():
        print("results:", sorted(p.name for p in res.iterdir()))
    rep = Path("results/report.md")
    if rep.is_file():
        text = rep.read_text(encoding="utf-8", errors="replace")
        print(text[:4000])
        if len(text) > 4000:
            print("\n... [truncated]")
else:
    print("RUN_TRAINING is False, no report to build.")



Wrote stub report: results/report.md
Missing: results/summary_metrics.csv results/per_split_seed_metrics.csv
per_split_seed_metrics.csv  report.md  runs
# Pancreatic MatchMaker report

## Error

Missing input CSV(s). Training likely failed or did not finish.

- Expected: `results/summary_metrics.csv`
- Expected: `results/per_split_seed_metrics.csv`

Fix NaN labels / rerun training, then rebuild report.


In [ ]:
# 10) Optional: copy outputs to Drive (set SAVE_TO_DRIVE + RUN_TRAINING in knobs cell)
from pathlib import Path
import os
import shutil

ROOT = Path(os.environ.get("MATCHMAKER_ROOT", ".")).resolve()

if SAVE_TO_DRIVE and RUN_TRAINING:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
    tgt = DRIVE_BACKUP_ROOT / "results"
    if tgt.exists():
        shutil.rmtree(tgt)
    shutil.copytree(ROOT / "results", tgt)
    print("Copied results to", tgt)

    stgt = DRIVE_BACKUP_ROOT / "splits"
    if stgt.exists():
        shutil.rmtree(stgt)
    shutil.copytree(ROOT / "splits", stgt)
    print("Copied splits to", stgt)
else:
    print("Drive copy skipped.")


## Troubleshooting

- **GitHub:** Edit `GIT_REPO_URL` / `GIT_BRANCH` in **Load codebase**. Private repo: `https://TOKEN@github.com/org/repo.git`.
- **Data:** Put `drug1_chem.csv`, `drug2_chem.csv`, `cell_line_gex.csv`, and `synergy - comb - Combination data.csv` in **My Drive** (root is fine), then run **Copy data from Google Drive** — names must match exactly.
- **Git / cwd:** If clone says it cannot read cwd, run the **cleanup** cell, then **Load codebase** again.
- **OOM:** Lower `BATCH_SIZE` in knobs.
- **Legacy recipe:** `NORM = 'tanh_norm'`, `WEIGHT_MODE = 'log'` in knobs before training.
